In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
        .appName("teste_azure_definitivo")
        .master("local[*]")
        
        # O Spark vai baixar o Azure e todas as 22 dependências (incluindo o Jetty) no mesmo nível!
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-azure:3.3.4,com.microsoft.azure:azure-storage:8.6.6")
        
        # Configurações do Azurite (Conta padrão)
        .config("spark.hadoop.fs.azure.account.key.devstoreaccount1.blob.core.windows.net", "Eby8vdM02xNOcqFlqUwJPLlmEtlCDXJ1OUzFT50uSRZ6IFsuFq2UVErCz4I6tq/K1SZFPTOtr/KBHBeksoGMGw==")
        .config("spark.hadoop.fs.azure.use.https", "false")
        .config("fs.azure.use.https", "false")
        
        # Delta Lake (Já está embutido na imagem via Dockerfile)
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        
        .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-azure added as a dependency
com.microsoft.azure#azure-storage added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f8a96a03-6a99-4ddf-bef5-068f01095ae0;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-guava;1.1.1 in central
	found org.eclipse.jetty#jetty-util-ajax;9.4.43.v20210629 in central
	found org.eclipse.jetty#jetty-util;9.4.43.v20210629 in central
	found org.codehaus.jackson#jackson-mapper-asl;1.9.13 in central
	found org.codehaus.jackson#jackson-core-asl;1.9.13 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.F

In [3]:
csv_path = "wasb://project@devstoreaccount1.blob.core.windows.net/raw/Produto.csv"

df_cl = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(csv_path)
)

df_cl.show()

+---+--------------------+-----+-----------+
| id|                name|price|category_id|
+---+--------------------+-----+-----------+
|  0|              Sapato|   96|          0|
|  1|               Bolsa|   21|          0|
|  2|               Calça|   75|          0|
|  3|              Camisa|   70|          0|
|  4|             Vestido|   57|          0|
|  5|             Perfume|   29|          1|
|  6|    Creme de Barbear|   15|          1|
|  7|          Hidratante|   10|          1|
|  8|             Shampoo|   15|          1|
|  9|      Protetor Solar|   11|          1|
| 10|           Geladeira| 1952|          2|
| 11|               Fogão| 1373|          2|
| 12|                  TV| 1992|          2|
| 13|    Máquina de Lavar| 1826|          2|
| 14|    Máquina de Secar| 1774|          2|
| 15| O Romance Exagerado|   46|          3|
| 16|     Suspense Demais|   42|          3|
| 17|Terror Aterrorizante|   45|          3|
| 18| A Comédia Engraçada|   20|          3|
| 19|   A 

In [6]:
path_destino = "wasb://project@devstoreaccount1.blob.core.windows.net/bronze/Produto"

df_cl.write\
    .format("delta")\
    .mode("overwrite")\
    .save(path_destino)

26/07/21 18:08:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [7]:
spark.sql("""
        DESCRIBE DETAIL
        delta.`wasb://project@devstoreaccount1.blob.core.windows.net/bronze/Produto`
          
        """).show(truncate=False)

+------+------------------------------------+----+-----------+--------------------------------------------------------------------+-----------------------+-------------------+----------------+--------+-----------+----------+----------------+----------------+------------------------+
|format|id                                  |name|description|location                                                            |createdAt              |lastModified       |partitionColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|tableFeatures           |
+------+------------------------------------+----+-----------+--------------------------------------------------------------------+-----------------------+-------------------+----------------+--------+-----------+----------+----------------+----------------+------------------------+
|delta |79837880-ce4c-4741-8ff6-851d6c865ffb|NULL|NULL       |wasb://project@devstoreaccount1.blob.core.windows.net/bronze/Produto|2026-07-21 18:08:

In [8]:
df_delta =(
    spark.read
    .format("delta")
    .load("wasb://project@devstoreaccount1.blob.core.windows.net/bronze/Produto")
)
df_delta.show()

+---+--------------------+-----+-----------+
| id|                name|price|category_id|
+---+--------------------+-----+-----------+
|  0|              Sapato|   96|          0|
|  1|               Bolsa|   21|          0|
|  2|               Calça|   75|          0|
|  3|              Camisa|   70|          0|
|  4|             Vestido|   57|          0|
|  5|             Perfume|   29|          1|
|  6|    Creme de Barbear|   15|          1|
|  7|          Hidratante|   10|          1|
|  8|             Shampoo|   15|          1|
|  9|      Protetor Solar|   11|          1|
| 10|           Geladeira| 1952|          2|
| 11|               Fogão| 1373|          2|
| 12|                  TV| 1992|          2|
| 13|    Máquina de Lavar| 1826|          2|
| 14|    Máquina de Secar| 1774|          2|
| 15| O Romance Exagerado|   46|          3|
| 16|     Suspense Demais|   42|          3|
| 17|Terror Aterrorizante|   45|          3|
| 18| A Comédia Engraçada|   20|          3|
| 19|   A 

In [12]:
spark.stop()

In [11]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
+---------+

